In [1]:
import pandas as pd
import torch
import os
import numpy as np
from torchvision.io import decode_image, ImageReadMode
from torch.utils.data import Dataset, random_split, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision.transforms import v2
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score


In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [3]:
labelingData = pd.read_csv("HealthcareData\Data_Entry_2017.csv")
labelingDataCorrected = labelingData[["Image Index", "Finding Labels"]]
for label in labelingDataCorrected["Finding Labels"].unique():
    if("|" not in label):
        print(label)

Cardiomegaly
No Finding
Hernia
Infiltration
Nodule
Emphysema
Effusion
Atelectasis
Pleural_Thickening
Pneumothorax
Mass
Fibrosis
Consolidation
Edema
Pneumonia


In [4]:
dataset_dir = "C:/Users/harri/Desktop/Github/Breast Disease Detection/HealthcareData/diseaseImages"

In [5]:
class BreastDiseaseDataset(Dataset):
    def __init__(self, annotations, img_dir, transform=None, target_transform=None):
        self.img_labels = annotations
        self.img_dir = img_dir
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        diseaseTranslation = {"No Finding":0, "Atelectasis":1, "Consolidation":2, "Infiltration": 3, "Pneumothorax": 4, "Edema": 5, "Emphysema": 6, "Fibrosis": 7, "Effusion": 8, "Pneumonia": 9, "Pleural_Thickening": 10, "Cardiomegaly": 11, "Mass": 12, "Hernia": 13, "Nodule": 14}
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0]).replace("\\", "/")
        image = decode_image(img_path, mode=ImageReadMode.GRAY).float() / 255.0
        labels = str(self.img_labels[self.img_labels["Image Index"] == self.img_labels.iloc[idx, 0]]["Finding Labels"].iloc[0])
        labels = labels.split("|")
        correctLabels = np.zeros(15, dtype=int)
        for label in labels:
            correctLabels[diseaseTranslation[label]] = 1
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            labels = self.target_transform(labels)
        return image, correctLabels

In [6]:
totalDataset = BreastDiseaseDataset(labelingDataCorrected, dataset_dir)

In [7]:
totalDataset.__getitem__(12)

(tensor([[[0.0196, 0.0314, 0.0314,  ..., 0.0314, 0.0353, 0.0196],
          [0.0353, 0.0667, 0.0627,  ..., 0.0588, 0.0627, 0.0392],
          [0.0353, 0.0627, 0.0588,  ..., 0.0588, 0.0588, 0.0431],
          ...,
          [0.0392, 0.0706, 0.0745,  ..., 0.2627, 0.2902, 0.1804],
          [0.0196, 0.0353, 0.0353,  ..., 0.1176, 0.1333, 0.0824],
          [0.0039, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]]]),
 array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1]))

In [8]:
#for i in range(70):
 #   print(len(totalDataset.__getitem__(i)))

In [9]:
train_dataset, test_dataset = random_split(totalDataset, [0.8, 0.2])

In [10]:
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=True)

In [11]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1,5,13)
        #self.norm = nn.BatchNorm1d(3078507)
        self.pool = nn.MaxPool2d(4, stride=2)
        self.prelu1 = nn.PReLU() 
        self.fc1 = nn.Linear(1275125, 1024)
        self.drop = nn.Dropout(p=0.5)
        self.fc3 = nn.Linear(1024, 15)

    def forward(self, x):
        x = self.conv1(x)
        x = self.pool(x)
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = self.prelu1(x)
        #x = F.relu(self.norm(x))
        #x = self.prelu1(x)
        x = F.leaky_relu(self.fc1(x))
        x = F.leaky_relu(self.drop(x))
        x = self.fc3(x)
        return x

In [12]:
net = Net()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)
#optimizer = optim.AdamW(net.parameters(), lr=0.001)

In [14]:
for epoch in range(2):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(train_dataloader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels.float())
        print(loss)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 2000 == 1999:    # print every 2000 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

print('Finished Training')

RuntimeError: Input type (torch.FloatTensor) and weight type (torch.cuda.FloatTensor) should be the same or input should be a MKLDNN tensor and weight is a dense tensor

In [65]:
# 2. Switch the model to evaluation mode
net.eval()

# Optional: Create lists to store predictions and true labels
all_predictions = []
all_targets = []

with torch.no_grad():
    for batch in test_dataloader:
        # Unpack the batch (assumes your dataloader yields inputs and targets)
        inputs, targets = batch
        outputs = net(inputs)
        all_predictions.append(outputs)
        all_targets.append(targets)

In [70]:
final_auc = roc_auc_score(np.concatenate(all_targets), np.concatenate(all_predictions))
print(f"Dataset ROC-AUC: {final_auc:.4f}")

Dataset ROC-AUC: 0.5872


In [13]:
#loading model option
net = Net().to(device)
net.load_state_dict(torch.load("model.pth", weights_only=True))

<All keys matched successfully>

In [62]:
torch.save(net.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth
